# COE Scenario Generation
Generate creative scenarios for error chains using GPT-4o batch API.


In [1]:
import os
import sys
import json
from pathlib import Path
from tokens import openai_key

NOTEBOOK_ROOT = Path("/scratch/jq2uw/MME/instruct_vlm_edit")
os.chdir(NOTEBOOK_ROOT)
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.append(str(NOTEBOOK_ROOT))

from revlm.metrics.utils.e_gen_scenario import COEScenarioGenerator


In [2]:
# Initialize generator
MODEL_NAME = "Qwen3-VL-8B-Instruct"
# MODEL_NAME = "Qwen3-VL-4B-Instruct"
# MODEL_NAME = "llava-1.5-7b-hf"
# MODEL_NAME = "instructblip-vicuna-7b"

# DATASET = "fvqa"  
DATASET = "aokvqa" 

gen = COEScenarioGenerator(
    dataset_name=DATASET,
    model_name=MODEL_NAME,
    openai_key=openai_key
)


COEScenarioGenerator: Qwen3-VL-8B-Instruct/aokvqa, 20 batches


## Step 1: Load COE results and prepare batch input


In [3]:
# Load COE prediction results
coe_path = f"results/pred_postedit/baseline/{MODEL_NAME}/{DATASET}/coe_prediction.json"
with open(coe_path, "r") as f:
    coe_results = json.load(f)

print(f"Loaded {len(coe_results)} COE results from {coe_path}")

# Check how many have error chains
with_errors = [r for r in coe_results if gen._get_error_chains(r)]
print(f"Samples with error chains: {len(with_errors)}/{len(coe_results)}")

# Preview a few
for r in with_errors[:3]:
    chains = gen._get_error_chains(r)
    print(f"uid={r['uid']}: {len(chains)} error chains")

# Preview the prompt for one sample
sample = with_errors[1]
chains = gen._get_error_chains(sample)
print(f"Sample has {len(chains)} error chains. Showing first:\n")
print("=== PROMPT ===")
print(gen.format_prompt(chains[0]["chain"]))

Loaded 7139 COE results from results/pred_postedit/baseline/Qwen3-VL-8B-Instruct/aokvqa/coe_prediction.json
Samples with error chains: 3195/7139
uid=1: 7 error chains
uid=14: 4 error chains
uid=19: 12 error chains
Sample has 4 error chains. Showing first:

=== PROMPT ===
Given these visual facts:
"The image shows the name of a theater and the date 2009. When did the namesake of this theater die is 2009."

Generate 3 different creative scenarios where ALL these facts would be visually true.

Requirements:
- Each scenario must be exactly one sentence (less than 20 words)
- Be creative but plausible
- Describe what would be visible in the image
- Do not contradict the given facts

Examples:
Visual facts: "A person is standing on a board. There are waves around."
1. A surfer rides a wave at a tropical beach during sunset.
2. A wakeboarder is pulled behind a speedboat on a calm lake.
3. A paddleboarder balances on ocean waters near a rocky coastline.

Visual facts: "The cake has multiple ti

In [4]:
# Generate batch request files (uncomment to run)
gen.run_input(coe_results, max_sentences=None)  # or max_sentences=5 to filter


Created 14373 requests in data/coe_gen/Qwen3-VL-8B-Instruct/aokvqa/requests/batch.jsonl


## Step 2: Test with ONE batch before firing all


In [5]:
# Check batch files created
batch_files = list(gen.batch_dir.glob("batch_*.jsonl"))
print(f"Batch files: {len(batch_files)}")
for bf in sorted(batch_files)[:5]:
    with open(bf) as f:
        n_lines = len(f.readlines())
    print(f"  {bf.name}: {n_lines} requests")
    
# Preview first request in batch_0
batch_0 = gen.batch_dir / "batch_0.jsonl"
if batch_0.exists():
    with open(batch_0) as f:
        first_req = json.loads(f.readline())
    print(json.dumps(first_req, indent=2)[:1000])  # truncate if too long



Batch files: 20
  batch_0.jsonl: 719 requests
  batch_1.jsonl: 719 requests
  batch_10.jsonl: 719 requests
  batch_11.jsonl: 719 requests
  batch_12.jsonl: 719 requests
{
  "custom_id": "1_[2]",
  "method": "POST",
  "url": "/v1/chat/completions",
  "body": {
    "model": "gpt-4o-mini",
    "messages": [
      {
        "role": "system",
        "content": "You generate creative visual scenarios from given facts."
      },
      {
        "role": "user",
        "content": "Given these visual facts:\n\"The man would not have luggage waiting for a delivery on the street. What is the man by the bags awaiting is cab.\"\n\nGenerate 3 different creative scenarios where ALL these facts would be visually true.\n\nRequirements:\n- Each scenario must be exactly one sentence (less than 20 words)\n- Be creative but plausible\n- Describe what would be visible in the image\n- Do not contradict the given facts\n\nExamples:\nVisual facts: \"A person is standing on a board. There are waves around.\"\n

In [6]:
# Submit ONLY batch 0 first (uncomment to run)
gen._run_request_batch(0)


In [7]:
# Get batch 0 results (after it completes)
try:
    results_0 = gen._get_response_batch(0)
    print(f"Got {len(results_0)} results from batch 0")
    
    # Preview first few
    for r in results_0[:3]:
        print(f"\nuid={r['uid']}:")
        for i, s in enumerate(r['scenarios'], 1):
            print(f"  {i}. {s}")
except Exception as e:
    print(f"Error: {e}")


Got 719 results from batch 0

uid=1:
  1. A businessman stands at a bustling airport curb, tapping his smartwatch as taxis zoom by. Dressed in a sharp suit, he glances at a stylish set of bags lined up next to him, ready for a cab that’s pulling up.
  2. A traveler waits patiently at a busy city street corner, surrounded by skyscrapers and vibrant billboards. He checks his phone, while sleek leather bags rest against the pavement, waiting for the cab he just summoned.
  3. A prospective student stands outside a university campus with historic buildings in view. In front of him, a pair of vibrant duffel bags sit expectantly on the sidewalk as he looks out for a cab amidst the bustling crowd of other new arrivals.

uid=1:
  1. A busy urban street is bustling with people, while a skateboarder performs tricks on the sidewalk oblivious to a man standing beside a stack of luggage waiting for his cab. Skyscrapers loom in the background, with a taxi waiting at the curb.
  2. In a sunny park, a

## Step 3: Fire all batches (after testing batch 0)


In [8]:
# Submit all remaining batches (uncomment to run)
gen.run_request()


In [9]:
# Check status of all submitted batches
for b in range(gen.n_batches):
    meta_path = gen.meta_dir / f"meta_{b}.json"
    if meta_path.exists():
        with open(meta_path) as f:
            meta = json.load(f)
        try:
            job = gen.client.batches.retrieve(meta["job_id"])
            print(f"Batch {b}: {job.status}")
        except Exception as e:
            print(f"Batch {b}: error - {e}")


Batch 0: completed
Batch 1: completed
Batch 2: completed
Batch 3: completed
Batch 4: completed
Batch 5: completed
Batch 6: completed
Batch 7: completed
Batch 8: completed
Batch 9: completed
Batch 10: completed
Batch 11: completed
Batch 12: completed
Batch 13: completed
Batch 14: completed
Batch 15: completed
Batch 16: completed
Batch 17: completed
Batch 18: completed
Batch 19: completed


In [10]:
# Resubmit failed batches if needed (uncomment and modify list)
# gen.resubmit_request([0, 1, 2])  # list of batch indices to resubmit


## Step 4: Get all results


In [11]:
# Get all scenarios (after all batches complete)
all_scenarios = gen.get_scenarios()
print(f"Total scenarios: {len(all_scenarios)}")

# Preview results
for r in all_scenarios[:3]:
    print(f"\nuid={r['uid']}, indices={r['indices']}:")
    for i, s in enumerate(r['scenarios'], 1):
        print(f"  {i}. {s}")


Total scenarios: 14373

uid=1, indices=[2]:
  1. A businessman stands at a bustling airport curb, tapping his smartwatch as taxis zoom by. Dressed in a sharp suit, he glances at a stylish set of bags lined up next to him, ready for a cab that’s pulling up.
  2. A traveler waits patiently at a busy city street corner, surrounded by skyscrapers and vibrant billboards. He checks his phone, while sleek leather bags rest against the pavement, waiting for the cab he just summoned.
  3. A prospective student stands outside a university campus with historic buildings in view. In front of him, a pair of vibrant duffel bags sit expectantly on the sidewalk as he looks out for a cab amidst the bustling crowd of other new arrivals.

uid=1, indices=[3]:
  1. A busy urban street is bustling with people, while a skateboarder performs tricks on the sidewalk oblivious to a man standing beside a stack of luggage waiting for his cab. Skyscrapers loom in the background, with a taxi waiting at the curb.
  2

In [12]:
all_scenarios

[{'uid': '1',
  'indices': [2],
  'scenarios': ['A businessman stands at a bustling airport curb, tapping his smartwatch as taxis zoom by. Dressed in a sharp suit, he glances at a stylish set of bags lined up next to him, ready for a cab that’s pulling up.',
   'A traveler waits patiently at a busy city street corner, surrounded by skyscrapers and vibrant billboards. He checks his phone, while sleek leather bags rest against the pavement, waiting for the cab he just summoned.',
   'A prospective student stands outside a university campus with historic buildings in view. In front of him, a pair of vibrant duffel bags sit expectantly on the sidewalk as he looks out for a cab amidst the bustling crowd of other new arrivals.'],
  'raw': '1. A businessman stands at a bustling airport curb, tapping his smartwatch as taxis zoom by. Dressed in a sharp suit, he glances at a stylish set of bags lined up next to him, ready for a cab that’s pulling up.\n\n2. A traveler waits patiently at a busy ci